## Phase 2 - Feature Engineering
**Goal:** Transform raw ECG signal into structured, learnable features for the ML model.

Raw ECG = 650,000 voltage readings - too noisy for a model to learn from directly.
Features extracted:
- **RR Interval** - time between consecutive heartbeats (ms)
- **Heart Rate** - beats per minute derived from RR interval
- **RR Diff** - beat-to-beat change in RR interval (captures sudden rhythm shifts)

In [1]:
import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

C:\Users\Manoj\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# Load the same record
record = wfdb.rdrecord('100', pn_dir='mitdb')
annotation = wfdb.rdann('100', 'atr', pn_dir='mitdb')

In [3]:
ecg_signal = record.p_signal[:, 0]
fs = record.fs

In [4]:
ann_samples = annotation.sample
ann_symbols = annotation.symbol

print(f"Signal loaded - {len(ecg_signal)} samples at {fs} Hz")

Signal loaded - 650000 samples at 360 Hz


In [5]:
# Calculate RR intervals - time between consecutive beats
rr_intervals = np.diff(ann_samples)

# Convert from samples to milliseconds
rr_ms = (rr_intervals / fs) * 1000

print(f"Total RR intervals: {len(rr_ms)}")
print(f"Mean RR interval: {np.mean(rr_ms):.2f} ms")
print(f"Min RR interval:  {np.min(rr_ms):.2f} ms")
print(f"Max RR interval:  {np.max(rr_ms):.2f} ms")

Total RR intervals: 2273
Mean RR interval: 794.32 ms
Min RR interval:  163.89 ms
Max RR interval:  1130.56 ms


In [6]:
# Heart rate from RR interval
heart_rate = 60000 / rr_ms  # 60,000 ms in a minute

print(f"Mean Heart Rate: {np.mean(heart_rate):.2f} bpm")
print(f"Min Heart Rate:  {np.min(heart_rate):.2f} bpm")
print(f"Max Heart Rate:  {np.max(heart_rate):.2f} bpm")

Mean Heart Rate: 75.94 bpm
Min Heart Rate:  53.07 bpm
Max Heart Rate:  366.10 bpm


In [7]:
# Build a feature table - one row per beat
feature_df = pd.DataFrame({
    'sample':       ann_samples[1:],      # beat location
    'symbol':       ann_symbols[1:],      # beat type label
    'rr_interval':  rr_ms,                # RR interval in ms
    'heart_rate':   heart_rate,           # beats per minute
    'rr_diff':      np.append(0, np.diff(rr_ms)),  # change from previous RR
})

print(feature_df.head(10))
print(f"Shape: {feature_df.shape}")

   sample symbol  rr_interval  heart_rate     rr_diff
0      77      N   163.888889  366.101695    0.000000
1     370      N   813.888889   73.720137  650.000000
2     662      N   811.111111   73.972603   -2.777778
3     946      N   788.888889   76.056338  -22.222222
4    1231      N   791.666667   75.789474    2.777778
5    1515      N   788.888889   76.056338   -2.777778
6    1809      N   816.666667   73.469388   27.777778
7    2044      A   652.777778   91.914894 -163.888889
8    2402      N   994.444444   60.335196  341.666667
9    2706      N   844.444444   71.052632 -150.000000
Shape: (2273, 5)


## Key Findings from Feature Table

**Row 0** - RR interval of 163ms / 366 bpm is not a real beat.
It is an artifact of the '+' rhythm marker annotation at the start of the record.
This will be handled during model training.

**Row 7** - symbol = A (Premature Atrial Contraction)
- RR interval dropped to 652ms - beat arrived early
- rr_diff = -163ms - sudden change from the previous beat
The features are already capturing the anomaly visually before the model has seen anything.
This is exactly the signal Isolation Forest will learn to flag.

**Shape: (2273, 5)** - 2273 beats, 5 features each. Ready for modelling.